This notebook scrapes NVMI data off da web 

In [8]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from tqdm import tqdm

Scraping script

In [10]:
# Config
BASE_URL = "https://globalland.vito.be/download/netcdf/ndvi/ndvi_1km_v3_10daily/"
START_YEAR = 2004
OUTPUT_DIR = "../../../../data/finaldatasets/covariates/Covariates/NVMI/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Step 1: Get year folders from root ---
resp = requests.get(BASE_URL)
soup = BeautifulSoup(resp.text, "html.parser")

year_urls = [
    urljoin(BASE_URL, link.get("href"))
    for link in soup.find_all("a")
    if link.get("href") and link.get("href").endswith("/")
    and link.get("href")[:4].isdigit()
    and int(link.get("href")[:4]) >= START_YEAR
]

# --- Step 2: For each year folder (2004+) ---
for year_url in year_urls:
    year_str = os.path.basename(urlparse(year_url.rstrip("/")).path)
    year_out_dir = os.path.join(OUTPUT_DIR, year_str)
    os.makedirs(year_out_dir, exist_ok=True)
    print(f"\n📁 Accessing year: {year_url}")

    # Get 10-daily subfolders (e.g., 20040101/)
    sub_resp = requests.get(year_url)
    sub_soup = BeautifulSoup(sub_resp.text, "html.parser")
    ten_day_urls = [
        urljoin(year_url, link.get("href"))
        for link in sub_soup.find_all("a")
        if link.get("href") and link.get("href").endswith("/") and link.get("href").startswith(year_str)
    ]

    # --- Step 3: For each 10-day subfolder, download .nc files ---
    for ten_day_url in ten_day_urls:
        ten_day_str = os.path.basename(urlparse(ten_day_url.rstrip("/")).path)
        ten_day_dir = os.path.join(year_out_dir, ten_day_str)
        os.makedirs(ten_day_dir, exist_ok=True)
        print(f"📂 Checking: {ten_day_url}")

        file_resp = requests.get(ten_day_url)
        file_soup = BeautifulSoup(file_resp.text, "html.parser")

        nc_files = [
            link.get("href") for link in file_soup.find_all("a")
            if link.get("href") and link.get("href").endswith(".nc")
        ]

        for nc_file in tqdm(nc_files, desc=f"⬇ Downloading {ten_day_str}"):
            file_url = urljoin(ten_day_url, nc_file)
            file_name = os.path.basename(nc_file)
            out_path = os.path.join(ten_day_dir, file_name)

            if not os.path.exists(out_path):
                r = requests.get(file_url)
                if r.status_code == 200:
                    with open(out_path, "wb") as f:
                        f.write(r.content)
                else:
                    tqdm.write(f"❌ Failed: {file_name} (HTTP {r.status_code})")
            else:
                tqdm.write(f"✔️ Skipped (exists): {file_name}")


📁 Accessing year: https://globalland.vito.be/download/netcdf/ndvi/ndvi_1km_v3_10daily/2004/
📂 Checking: https://globalland.vito.be/download/netcdf/ndvi/ndvi_1km_v3_10daily/2004/20040101/


⬇ Downloading 20040101: 100%|██████████| 1/1 [00:37<00:00, 37.57s/it]


📂 Checking: https://globalland.vito.be/download/netcdf/ndvi/ndvi_1km_v3_10daily/2004/20040111/


⬇ Downloading 20040111: 100%|██████████| 1/1 [00:37<00:00, 37.40s/it]


📂 Checking: https://globalland.vito.be/download/netcdf/ndvi/ndvi_1km_v3_10daily/2004/20040121/


...

⬇ Downloading 20200601: 100%|██████████| 1/1 [01:44<00:00, 104.10s/it]


📂 Checking: https://globalland.vito.be/download/netcdf/ndvi/ndvi_1km_v3_10daily/2020/20200611/


⬇ Downloading 20200611: 100%|██████████| 1/1 [01:35<00:00, 95.33s/it]


📂 Checking: https://globalland.vito.be/download/netcdf/ndvi/ndvi_1km_v3_10daily/2020/20200621/


⬇ Downloading 20200621: 100%|██████████| 1/1 [01:35<00:00, 95.31s/it]
